In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
#  DASHBOARD DINÁMICO — MERCADO LABORAL MANUFACTURERO
#  San Luis Potosí y Región Centro-Norte | 2018–2025
#
#  INSTRUCCIONES PARA JUPYTERLAB:
#  1. Instala dependencias (solo la primera vez), en una celda nueva:
#       !pip install ipympl ipywidgets
#  2. Reinicia el Kernel: Kernel → Restart Kernel
#  3. Ejecuta PRIMERO en una celda separada:
#       %matplotlib inline
#  4. En la siguiente celda pega y ejecuta este script completo
# ╚══════════════════════════════════════════════════════════════════╝

import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import io, base64

plt.ioff()

# ── RUTA DEL ARCHIVO ─────────────────────────────────────────
RUTA_ARCHIVO = r'C:\Users\USUARIO\3D Objects\Untitled Folder\Base de datos EQUIPO 2.xlsx'

if not os.path.exists(RUTA_ARCHIVO):
    raise FileNotFoundError(
        f"\n¡Archivo no encontrado en:\n{RUTA_ARCHIVO}\n"
        "Verifica la ruta o ejecuta en una celda:\n"
        "  import os\n"
        "  for r,_,fs in os.walk(os.path.expanduser('~')):\n"
        "    for f in fs:\n"
        "      if 'EQUIPO' in f: print(os.path.join(r,f))"
    )

# ── PALETA ───────────────────────────────────────────────────
C1,C2,C3,C4,C5 = '#1B3A6B','#C8372D','#2E7D32','#F57F17','#6A1B9A'
BG, CARD        = '#F4F6F9','#FFFFFF'

# ── CARGA DE DATOS ───────────────────────────────────────────
df = pd.read_excel(RUTA_ARCHIVO, sheet_name='Base de datos')

months_es = {'Ene':1,'Feb':2,'Mar':3,'Abr':4,'May':5,'Jun':6,
             'Jul':7,'Ago':8,'Sep':9,'Oct':10,'Nov':11,'Dic':12}

def parse_fecha(s):
    m, y = s.split()
    return pd.Timestamp(year=int(y), month=months_es[m], day=1)

df['fecha'] = df['Fecha'].apply(parse_fecha)
df = df.sort_values(['Entidad','fecha']).reset_index(drop=True)

ENTIDADES   = ['San Luis Potosi','Aguascalientes','Queretaro','Guanajuato']
COLORES_ENT = dict(zip(ENTIDADES,[C1,C2,C3,C4]))

def prep(sub):
    s = sub.copy().sort_values('fecha').reset_index(drop=True)
    s['salario_nominal'] = s['Remuneraciones pagadas '] / s['Personal ocupado total ']
    s['salario_real']    = (s['salario_nominal'] / s['INPC (mensual)']) * 100
    base = s.loc[s['fecha'].dt.year==2018,'Personal ocupado total '].mean()
    s['ind_personal']    = (s['Personal ocupado total '] / base) * 100
    s['crec_personal']   = s['Personal ocupado total '].pct_change(12) * 100
    s['ln_salario_real'] = np.log(s['salario_real'])
    s['ln_personal']     = np.log(s['Personal ocupado total '])
    s['tendencia']       = np.arange(len(s))
    return s

DATOS = {e: prep(df[df['Entidad']==e]) for e in ENTIDADES}

# ── REGRESIÓN ────────────────────────────────────────────────
def calcular_regresion(entidad, fi, ff):
    d    = DATOS[entidad]
    mask = (d['fecha']>=fi)&(d['fecha']<=ff)
    reg  = d[mask].dropna(subset=['ln_salario_real','ln_personal']).copy()
    if len(reg) < 10: return None
    X   = sm.add_constant(reg[['ln_personal','tendencia']])
    y   = reg['ln_salario_real']
    ols = sm.OLS(y, X).fit()
    nw  = ols.get_robustcov_results(cov_type='HAC', maxlags=4)
    p   = np.array(nw.params)
    pv  = np.array(nw.pvalues)
    nm  = X.columns.tolist()
    i1  = nm.index('ln_personal')
    return dict(reg=reg, ols=ols, params=p, pvals=pv, names=nm,
                beta1=p[i1], pval_b1=pv[i1],
                rsq=ols.rsquared, rsq_adj=ols.rsquared_adj,
                dw=durbin_watson(ols.resid),
                adf_p=adfuller(ols.resid.values,autolag='AIC')[1],
                y=y, y_pred=ols.fittedvalues, resid=ols.resid,
                fval=ols.fvalue, fp=ols.f_pvalue, nobs=int(ols.nobs))

# ── HELPERS ──────────────────────────────────────────────────
def card(ax):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_color('#D0D7E3'); sp.set_linewidth(0.8)

def ttl(ax, txt, sub=''):
    ax.set_title(('$\\bf{'+txt+'}$')+(f'\n{sub}' if sub else ''),
                 fontsize=9, color=C1, pad=5, loc='left')

def kpi_box(ax, label, valor, sub, color):
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.03,0.05),0.94,0.90,
                 boxstyle="round,pad=0.04",
                 facecolor=color, edgecolor='none', alpha=0.93))
    ax.text(0.5,0.74,label,ha='center',va='center',fontsize=9, color='white',fontweight='bold')
    ax.text(0.5,0.45,valor,ha='center',va='center',fontsize=16,color='white',fontweight='bold')
    ax.text(0.5,0.17,sub,  ha='center',va='center',fontsize=7.5,color='#FFFFFFCC')

# ── FIGURA → IMAGEN HTML (funciona en cualquier entorno) ─────
def fig_a_html(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight', facecolor=BG)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;height:auto;">'

# ════════════════════════════════════════════════════════════
#  WIDGETS
# ════════════════════════════════════════════════════════════
st  = {'description_width':'145px'}
lay = widgets.Layout(width='310px')

w_entidad = widgets.Dropdown(
    options=ENTIDADES, value='San Luis Potosi',
    description='🏭 Entidad:', style=st, layout=lay)

w_comp = widgets.SelectMultiple(
    options=ENTIDADES, value=['San Luis Potosi','Guanajuato'],
    description='📊 Comparar:', style=st,
    layout=widgets.Layout(width='310px', height='95px'))

w_anio_ini = widgets.IntSlider(
    value=2018, min=2018, max=2024, step=1,
    description='📅 Año inicio:', style=st, layout=lay,
    continuous_update=False)

w_anio_fin = widgets.IntSlider(
    value=2025, min=2019, max=2025, step=1,
    description='📅 Año fin:', style=st, layout=lay,
    continuous_update=False)

w_variable = widgets.RadioButtons(
    options=[('Personal Ocupado',          'Personal ocupado total '),
             ('Salario Real (índice)',      'salario_real'),
             ('Salario Nominal (x trab.)', 'salario_nominal'),
             ('Índice de Crecimiento',     'ind_personal')],
    value='Personal ocupado total ',
    description='📈 Variable:', style=st,
    layout=widgets.Layout(width='330px'))

w_roll = widgets.IntSlider(
    value=12, min=3, max=24, step=1,
    description='🔄 Rolling (m):', style=st, layout=lay,
    continuous_update=False)

w_tipo = widgets.ToggleButtons(
    options=[('Línea 📈','line'),('Área 🏔️','area'),('Barra 📊','bar')],
    value='line', description='Gráfica:',
    style={'description_width':'70px','button_width':'95px'})

w_covid = widgets.Checkbox(
    value=True, description='Mostrar período COVID-19',
    layout=widgets.Layout(width='260px'))

w_btn_exportar = widgets.Button(
    description='💾 Exportar PNG', button_style='success',
    layout=widgets.Layout(width='160px', height='34px'))
w_export_lbl = widgets.Label(value='')

# Áreas de imagen (HTML widgets — no dependen del backend)
img_principal   = widgets.HTML(value='<p style="color:gray">Cargando...</p>')
img_regresion   = widgets.HTML(value='<p style="color:gray">Cargando...</p>')
img_comparativo = widgets.HTML(value='<p style="color:gray">Cargando...</p>')
img_resumen     = widgets.HTML(value='<p style="color:gray">Cargando...</p>')

tab = widgets.Tab(children=[img_principal, img_regresion,
                             img_comparativo, img_resumen])
for i,t in enumerate(['🏭 Principal','📐 Regresión','🗺️ Comparativo','📋 Resumen']):
    tab.set_title(i, t)

# ════════════════════════════════════════════════════════════
#  FUNCIONES DE DIBUJO
# ════════════════════════════════════════════════════════════

def dibujar_principal():
    entidad  = w_entidad.value
    comp     = list(w_comp.value)
    variable = w_variable.value
    tipo     = w_tipo.value
    fi       = pd.Timestamp(year=w_anio_ini.value, month=1,  day=1)
    ff       = pd.Timestamp(year=w_anio_fin.value, month=12, day=31)
    covid    = w_covid.value
    d        = DATOS[entidad]
    ds       = d[(d['fecha']>=fi)&(d['fecha']<=ff)]

    fig = plt.figure(figsize=(13,8), facecolor=BG)
    gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.55,wspace=0.38,
                            top=0.93,bottom=0.06,left=0.07,right=0.97)
    fig.suptitle(f'Dashboard Principal — {entidad}  |  '
                 f'{w_anio_ini.value}–{w_anio_fin.value}',
                 fontsize=12,fontweight='bold',color=C1,y=0.97)

    ult  = ds.iloc[-1]  if len(ds)>0  else d.iloc[-1]
    prev = ds.iloc[-13] if len(ds)>12 else ds.iloc[0]
    def pct(a,b): return (a/b-1)*100 if b!=0 else 0
    kpis = [
        ('Personal Ocupado',
         f"{ult['Personal ocupado total ']:,.0f}",
         f"Var. anual: {pct(ult['Personal ocupado total '],prev['Personal ocupado total ']):+.1f}%",C1),
        ('Salario Real (índice)',
         f"{ult['salario_real']:.1f}",
         f"Var. anual: {pct(ult['salario_real'],prev['salario_real']):+.1f}%",C3),
        ('Remuneraciones',
         f"${ult['Remuneraciones pagadas ']/1e6:.2f}M",
         f"Var. anual: {pct(ult['Remuneraciones pagadas '],prev['Remuneraciones pagadas ']):+.1f}%",C2),
    ]
    for col,(lbl,val,sub,col_) in enumerate(kpis):
        kpi_box(fig.add_subplot(gs[0,col]),lbl,val,sub,col_)

    ax = fig.add_subplot(gs[1,:]); card(ax)
    lbl_y = {'Personal ocupado total ':'Personas','salario_real':'Índice (2018=100)',
              'salario_nominal':'$ por trab.','ind_personal':'Índice (Ene18=100)'}[variable]
    for e in comp:
        dm=DATOS[e]; mk=(dm['fecha']>=fi)&(dm['fecha']<=ff)
        lw=2.5 if e==entidad else 1.2; al=1.0 if e==entidad else 0.55
        if   tipo=='line': ax.plot(dm[mk]['fecha'],dm[mk][variable],color=COLORES_ENT[e],lw=lw,alpha=al,label=e)
        elif tipo=='area':
            ax.fill_between(dm[mk]['fecha'],dm[mk][variable],alpha=0.25 if e==entidad else 0.12,color=COLORES_ENT[e])
            ax.plot(dm[mk]['fecha'],dm[mk][variable],color=COLORES_ENT[e],lw=lw,alpha=al,label=e)
        elif tipo=='bar': ax.bar(dm[mk]['fecha'],dm[mk][variable],color=COLORES_ENT[e],alpha=al,width=20,label=e)
    if covid:
        ax.axvspan(pd.Timestamp('2020-03-01'),pd.Timestamp('2020-09-01'),
                   color='gray',alpha=0.13,label='COVID-19')
    ax.set_ylabel(lbl_y,fontsize=9); ax.legend(fontsize=8,framealpha=0.8)
    ttl(ax,variable.strip(),'Comparativo regional')

    ax2 = fig.add_subplot(gs[2,:]); card(ax2)
    crec=ds['crec_personal'].dropna()
    ax2.bar(ds.loc[crec.index,'fecha'],crec,color=np.where(crec>=0,C3,C2),alpha=0.8,width=20)
    ax2.axhline(0,color='black',lw=0.8); ax2.set_ylabel('% var. anual',fontsize=9)
    ttl(ax2,'Crecimiento Anual del Empleo',f'{entidad}')
    plt.tight_layout()
    return fig

def dibujar_regresion():
    entidad = w_entidad.value
    fi = pd.Timestamp(year=w_anio_ini.value, month=1,  day=1)
    ff = pd.Timestamp(year=w_anio_fin.value, month=12, day=31)
    R  = calcular_regresion(entidad, fi, ff)

    fig = plt.figure(figsize=(13,9), facecolor=BG)
    if R is None:
        fig.text(0.5,0.5,'⚠️ Insuficientes observaciones.\nAmplía el rango de años.',
                 ha='center',va='center',fontsize=14,color=C2)
        return fig

    gs = gridspec.GridSpec(2,3,figure=fig,hspace=0.48,wspace=0.38,
                           top=0.92,bottom=0.08,left=0.07,right=0.97)
    sig=("★★★" if R['pval_b1']<0.01 else "★★" if R['pval_b1']<0.05
         else "★" if R['pval_b1']<0.10 else "n.s.")
    fig.suptitle(f'Regresión OLS-HAC — {entidad}\n'
                 f'β₁={R["beta1"]:.4f} {sig}  R²={R["rsq"]:.4f}  DW={R["dw"]:.3f}  N={R["nobs"]}',
                 fontsize=11,fontweight='bold',color=C1,y=0.97)

    ax=fig.add_subplot(gs[0,0]); card(ax)
    sc=ax.scatter(R['reg']['ln_personal'],R['reg']['ln_salario_real'],
                  c=np.arange(len(R['reg'])),cmap='Blues_r',s=28,alpha=0.85,zorder=3)
    plt.colorbar(sc,ax=ax,label='Tiempo →',pad=0.02)
    xr=np.linspace(R['reg']['ln_personal'].min(),R['reg']['ln_personal'].max(),100)
    yr=R['params'][0]+R['params'][1]*xr+R['params'][2]*np.linspace(0,len(R['reg'])-1,100)
    ax.plot(xr,yr,color=C2,lw=2,label=f"β₁={R['beta1']:.3f}")
    ax.set_xlabel('ln(Personal)',fontsize=9); ax.set_ylabel('ln(Sal.Real)',fontsize=9)
    ax.legend(fontsize=8); ttl(ax,'Log-Log Scatter','')

    ax=fig.add_subplot(gs[0,1]); card(ax)
    ax.plot(R['reg']['fecha'],np.exp(R['y'].values),color=C1,lw=2,label='Observado')
    ax.plot(R['reg']['fecha'],np.exp(R['y_pred'].values),color=C2,lw=1.8,ls='--',label='Ajustado')
    ax.fill_between(R['reg']['fecha'],np.exp(R['y'].values),np.exp(R['y_pred'].values),alpha=0.15,color=C4)
    ax.set_ylabel('Salario Real',fontsize=9); ax.legend(fontsize=8)
    ttl(ax,'Observado vs Predicho','')

    ax=fig.add_subplot(gs[0,2]); card(ax)
    ax.bar(R['reg']['fecha'],R['resid'],color=np.where(R['resid']>=0,C3,C2),alpha=0.75,width=20)
    ax.axhline(0,color='black',lw=0.8); ax.set_ylabel('Residuo',fontsize=9)
    ttl(ax,'Residuos OLS',f"DW={R['dw']:.3f}")

    ax=fig.add_subplot(gs[1,0]); card(ax)
    (osm,osr),(sl,ic,r)=stats.probplot(R['resid'],dist='norm')
    ax.scatter(osm,osr,color=C1,s=20,alpha=0.7)
    ax.plot(osm,sl*np.array(osm)+ic,color=C2,lw=1.5)
    ax.set_xlabel('Cuantiles teóricos',fontsize=9); ax.set_ylabel('Cuantiles obs.',fontsize=9)
    ttl(ax,'Q-Q Plot',f'r={r:.3f}')

    d=DATOS[entidad]; ds=d[(d['fecha']>=fi)&(d['fecha']<=ff)]
    ax=fig.add_subplot(gs[1,1]); card(ax)
    rc=ds['Personal ocupado total '].rolling(w_roll.value).corr(ds['salario_real'])
    ax.plot(ds['fecha'],rc,color=C5,lw=2)
    ax.axhline(0,color='gray',lw=0.8,ls='--')
    ax.fill_between(ds['fecha'],rc,0,where=rc>0, alpha=0.3,color=C3,label='+')
    ax.fill_between(ds['fecha'],rc,0,where=rc<=0,alpha=0.3,color=C2,label='−')
    ax.set_ylabel('Correlación',fontsize=9); ax.legend(fontsize=8)
    ttl(ax,f'Corr. Rodante ({w_roll.value}m)','')

    ax=fig.add_subplot(gs[1,2]); ax.axis('off'); ax.set_facecolor('#EEF2FB')
    txt=(f"  RESULTADOS OLS-HAC\n  {'─'*26}\n"
         f"  β₀={R['params'][0]:+.4f}  p={R['pvals'][0]:.4f}\n"
         f"  β₁={R['params'][1]:+.4f}  p={R['pvals'][1]:.4f} {sig}\n"
         f"  β₂={R['params'][2]:+.4f}  p={R['pvals'][2]:.4f}\n\n"
         f"  R²     = {R['rsq']:.4f}\n  R²-adj = {R['rsq_adj']:.4f}\n"
         f"  F      = {R['fval']:.3f}  p={R['fp']:.5f}\n"
         f"  DW     = {R['dw']:.4f}\n  ADF-p  = {R['adf_p']:.4f}\n"
         f"  N      = {R['nobs']}\n\n")
    txt+=f"  ✔ +1%emp→+{R['beta1']:.2f}%sal" if R['pval_b1']<0.05 and R['beta1']>0 else \
         f"  ⚠ +1%emp→{R['beta1']:.2f}%sal"  if R['pval_b1']<0.05 else "  ✖ β₁ no sig."
    ax.text(0.04,0.97,txt,transform=ax.transAxes,fontsize=8.5,va='top',
            fontfamily='monospace',color='#1B3A6B',
            bbox=dict(boxstyle='round',facecolor='white',alpha=0.8))
    ttl(ax,'Resultados','')
    plt.tight_layout(); return fig

def dibujar_comparativo():
    comp=list(w_comp.value); variable=w_variable.value
    fi=pd.Timestamp(year=w_anio_ini.value,month=1,day=1)
    ff=pd.Timestamp(year=w_anio_fin.value,month=12,day=31)
    lbl_y={'Personal ocupado total ':'Personas','salario_real':'Índice (2018=100)',
           'salario_nominal':'$ por trab.','ind_personal':'Índice (Ene18=100)'}[variable]

    fig,axes=plt.subplots(2,2,figsize=(13,8),facecolor=BG)
    fig.suptitle(f'Comparativo Regional — {w_anio_ini.value}–{w_anio_fin.value}',
                 fontsize=12,fontweight='bold',color=C1)

    ax=axes[0,0]; card(ax)
    for e in comp:
        d=DATOS[e]; mk=(d['fecha']>=fi)&(d['fecha']<=ff)
        ax.plot(d[mk]['fecha'],d[mk][variable],color=COLORES_ENT[e],lw=2,label=e)
    if w_covid.value:
        ax.axvspan(pd.Timestamp('2020-03-01'),pd.Timestamp('2020-09-01'),color='gray',alpha=0.13)
    ax.set_ylabel(lbl_y,fontsize=9); ax.legend(fontsize=7); ttl(ax,'Serie Temporal','')

    ax=axes[0,1]; card(ax)
    for e in comp:
        d=DATOS[e]; mk=(d['fecha']>=fi)&(d['fecha']<=ff)
        ds=d[mk][variable].dropna()
        if len(ds)==0: continue
        ax.plot(d[mk].loc[ds.index,'fecha'],(ds/ds.iloc[0]-1)*100,
                color=COLORES_ENT[e],lw=2,label=e)
    ax.axhline(0,color='gray',lw=0.8,ls='--')
    ax.set_ylabel('% cambio acumulado',fontsize=9); ax.legend(fontsize=7)
    ttl(ax,'Crecimiento Acumulado','Base = primer mes')

    ax=axes[1,0]; card(ax)
    data_b,lbl_b=[],[]
    for e in comp:
        d=DATOS[e]; mk=(d['fecha']>=fi)&(d['fecha']<=ff)
        v=d[mk][variable].dropna().values
        if len(v): data_b.append(v); lbl_b.append(e[:10])
    bp=ax.boxplot(data_b,labels=lbl_b,patch_artist=True,
                  medianprops=dict(color='black',lw=1.5))
    for patch,e in zip(bp['boxes'],comp):
        patch.set_facecolor(COLORES_ENT[e]); patch.set_alpha(0.7)
    ax.set_ylabel(lbl_y,fontsize=9); ax.tick_params(axis='x',labelsize=7)
    ttl(ax,'Distribución Boxplot','')

    ax=axes[1,1]; card(ax)
    ultimos=[(e,DATOS[e][(DATOS[e]['fecha']>=fi)&(DATOS[e]['fecha']<=ff)][variable].iloc[-1])
             for e in comp if len(DATOS[e][(DATOS[e]['fecha']>=fi)&(DATOS[e]['fecha']<=ff)])>0]
    if ultimos:
        ents_b,vals_b=zip(*ultimos)
        bars=ax.bar(range(len(ents_b)),vals_b,color=[COLORES_ENT[e] for e in ents_b],alpha=0.85)
        ax.set_xticks(range(len(ents_b)))
        ax.set_xticklabels([e[:12] for e in ents_b],fontsize=7,rotation=10)
        for bar,val in zip(bars,vals_b):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()*1.01,
                    f'{val:,.0f}',ha='center',va='bottom',fontsize=7)
        ax.set_ylabel(lbl_y,fontsize=9)
    ttl(ax,'Último Mes del Período','')
    plt.tight_layout(); return fig

def dibujar_resumen():
    entidad=w_entidad.value
    fi=pd.Timestamp(year=w_anio_ini.value,month=1,day=1)
    ff=pd.Timestamp(year=w_anio_fin.value,month=12,day=31)
    R=calcular_regresion(entidad,fi,ff)
    d=DATOS[entidad]; ds=d[(d['fecha']>=fi)&(d['fecha']<=ff)]

    fig,axes=plt.subplots(1,2,figsize=(13,6),facecolor=BG)
    fig.suptitle(f'Resumen — {entidad}  |  {w_anio_ini.value}–{w_anio_fin.value}',
                 fontsize=12,fontweight='bold',color=C1)

    ax=axes[0]; ax.axis('off'); ax.set_facecolor('#F0F4FC')
    desc=ds[['Personal ocupado total ','salario_real','salario_nominal','INPC (mensual)']].describe().round(2)
    tbl=ax.table(cellText=desc.values.tolist(),rowLabels=desc.index.tolist(),
                 colLabels=['Personal\nOcupado','Sal.\nReal','Sal.\nNominal','INPC'],
                 loc='center',cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1.1,1.6)
    for (r,c),cell in tbl.get_celld().items():
        if r==0 or c==-1: cell.set_facecolor(C1); cell.set_text_props(color='white',fontweight='bold')
        elif r%2==0: cell.set_facecolor('#E8EDF6')
    ttl(ax,'Estadísticas Descriptivas','')

    ax=axes[1]; ax.axis('off'); ax.set_facecolor('#F0F4FC')
    if R:
        sig=("★★★ p<0.01" if R['pval_b1']<0.01 else "★★ p<0.05"
             if R['pval_b1']<0.05 else "★ p<0.10" if R['pval_b1']<0.10 else "no sig.")
        if   R['pval_b1']<0.05 and R['beta1']>0: resp=(f"✅ SÍ: +1% empleo → +{R['beta1']:.3f}%\n   en salario real de {entidad}.")
        elif R['pval_b1']<0.05:                  resp=(f"⚠️ PARADOJA: +1% empleo → {R['beta1']:.3f}%\n   salario real (exceso oferta?).")
        else:                                     resp="❌ Sin evidencia estadística (β₁\n   no significativo al 5%)."
        txt=(f"PREGUNTA CENTRAL\n{'─'*34}\n"
             f"¿El crecimiento industrial\npresionó los salarios reales?\n\n"
             f"{resp}\n\n{'─'*34}\n"
             f"β₁ elasticidad = {R['beta1']:+.4f}\n"
             f"p-valor        = {R['pval_b1']:.4f}  {sig}\n"
             f"R²             = {R['rsq']:.4f}\n"
             f"R² ajustado    = {R['rsq_adj']:.4f}\n"
             f"Durbin-Watson  = {R['dw']:.4f}\n"
             f"ADF (residuos) = {R['adf_p']:.4f}\n"
             f"Observaciones  = {R['nobs']}")
    else:
        txt="⚠️ Insuficientes datos.\nAmplía el rango de años."
    ax.text(0.05,0.95,txt,transform=ax.transAxes,fontsize=9.5,va='top',
            fontfamily='monospace',color='#1B3A6B',
            bbox=dict(boxstyle='round,pad=0.7',facecolor='white',
                      alpha=0.88,edgecolor=C1,linewidth=1.5))
    ttl(ax,'Conclusión Econométrica','')
    plt.tight_layout(); return fig

# ════════════════════════════════════════════════════════════
#  ACTUALIZACIÓN: convierte figura → PNG base64 → HTML widget
# ════════════════════════════════════════════════════════════
DIBUJOS = [dibujar_principal, dibujar_regresion,
           dibujar_comparativo, dibujar_resumen]
IMGS    = [img_principal, img_regresion, img_comparativo, img_resumen]

def actualizar(_=None):
    for fn, img_widget in zip(DIBUJOS, IMGS):
        img_widget.value = '<p style="color:gray;padding:10px">⏳ Actualizando...</p>'
    for fn, img_widget in zip(DIBUJOS, IMGS):
        try:
            fig = fn()
            img_widget.value = fig_a_html(fig)
        except Exception as e:
            img_widget.value = f'<p style="color:red">❌ Error: {e}</p>'

for w in [w_entidad,w_comp,w_variable,w_tipo,
          w_anio_ini,w_anio_fin,w_roll,w_covid]:
    w.observe(actualizar, names='value')

def exportar(_):
    w_export_lbl.value='⏳ Exportando...'
    fi=pd.Timestamp(year=w_anio_ini.value,month=1,day=1)
    ff=pd.Timestamp(year=w_anio_fin.value,month=12,day=31)
    carpeta=os.path.join(os.path.expanduser('~'),'3D Objects','Untitled Folder')
    nombres=['principal','regresion','comparativo','resumen']
    for fn,nom in zip(DIBUJOS,nombres):
        fig=fn()
        ruta=os.path.join(carpeta,f"dashboard_{w_entidad.value.replace(' ','_')}_{nom}.png")
        fig.savefig(ruta,dpi=150,bbox_inches='tight',facecolor=BG)
        plt.close(fig)
    w_export_lbl.value='✔ 4 PNG guardados en Untitled Folder'
w_btn_exportar.on_click(exportar)

# ════════════════════════════════════════════════════════════
#  LAYOUT FINAL
# ════════════════════════════════════════════════════════════
header = widgets.HTML("""
<div style="background:linear-gradient(90deg,#1B3A6B,#2E5FA3);
            padding:12px 20px;border-radius:10px;margin-bottom:8px;">
  <span style="color:white;font-size:17px;font-weight:bold;">
    📊 DASHBOARD LABORAL MANUFACTURERO — SLP Y REGIÓN
  </span><br>
  <span style="color:#BDD4F0;font-size:11px;">
    INEGI-EMIM &amp; Banco de México | Ene 2018 – Dic 2025
  </span>
</div>""")

panel_ctrl = widgets.VBox([
    widgets.HTML("<b style='color:#1B3A6B;font-size:13px'>⚙️ CONTROLES</b>"),
    w_entidad, w_comp,
    widgets.HTML("<hr style='margin:3px 0;border-color:#D0D7E3'>"),
    w_anio_ini, w_anio_fin,
    widgets.HTML("<hr style='margin:3px 0;border-color:#D0D7E3'>"),
    w_variable,
    widgets.HTML("<hr style='margin:3px 0;border-color:#D0D7E3'>"),
    w_tipo, w_roll, w_covid,
    widgets.HTML("<hr style='margin:3px 0;border-color:#D0D7E3'>"),
    widgets.HBox([w_btn_exportar]), w_export_lbl,
], layout=widgets.Layout(width='345px',padding='10px',
                         border='1px solid #D0D7E3',overflow_y='auto'))

ui = widgets.VBox([
    header,
    widgets.HBox([panel_ctrl, tab],
                 layout=widgets.Layout(gap='10px',align_items='flex-start'))
])

display(ui)
actualizar()